---

# 📂 Geração de Dados Sintéticos: GlobalForce USA (2023-2025)

## 1. Objetivo do Script
Este script tem como finalidade gerar uma base de dados robusta e realista para o projeto de automação de relatórios da **GlobalForce**. Ele simula o comportamento de uma força de trabalho de grande escala nos Estados Unidos ao longo de **36 meses**, permitindo o teste de performance do pipeline ETL e a validação de dashboards executivos.

---

## 2. Parâmetros da Simulação
* **Escopo Geográfico:** Estados Unidos (divididos em 4 regiões do Censo: Northeast, Midwest, South e West).
* **Volume de Dados:** 100.000 colaboradores únicos.
* **Janela Temporal:** Janeiro de 2023 a Dezembro de 2025 (Snapshots mensais).
* **Moeda:** Dólar Americano (USD).
* **Total de Registros Estimados:** ~3,6 milhões de linhas.

---

## 3. Lógica de Negócio Aplicada
Para garantir que os KPIs (Turnover, Utilização e Custo) reflitam cenários reais, as seguintes regras foram implementadas:

1.  **Ciclo de Vida (Tenure):** As datas de contratação retroagem até 2020, permitindo cálculos de tempo de casa.
2.  **Turnover Mensal:** Simulamos uma taxa de desligamento distribuída aleatoriamente, onde o colaborador deixa de aparecer na base nos meses subsequentes à sua data de término.
3.  **Utilização de Capacidade:** Calculada com base em uma carga horária padrão de **160h/mês**, com variações simulando ociosidade (abaixo de 75%) e sobrecarga (acima de 100%).
4.  **Estrutura de Custos:** Salários baseados em cargos (*Seniority Levels*), com flutuações mensais para simular bônus ou variações operacionais.

---

## 4. Dicionário de Dados (Output)

| Campo | Descrição | Exemplo |
| :--- | :--- | :--- |
| `period` | Mês de referência do registro | 2024-01-01 |
| `employee_id` | Identificador único do colaborador | 100452 |
| `region` | Região macro dos EUA | Northeast |
| `state` | Estado de atuação | NY |
| `department` | Departamento funcional | Engineering |
| `role` | Nível de senioridade | Senior |
| `monthly_cost` | Custo total do colaborador no mês (USD) | 12450.50 |
| `planned_hours` | Horas contratadas para o período | 160.0 |
| `worked_hours` | Horas efetivamente registradas | 155.5 |
| `overtime_hours` | Horas extras (Worked - Planned) | 5.0 |
| `goal_achievement` | % de atingimento de metas do mês | 92.5 |
| `status_in_month` | Status do colaborador no fechamento do mês | Active / Terminated |

---


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# --- CONFIGURAÇÕES ---
N_EMPLOYEES = 100000
# Definindo o intervalo de 3 anos (36 meses)
MONTHS = pd.date_range(start="2023-01-01", end="2025-12-01", freq="MS")
FILE_NAME = "globalforce_usa_3years_2023_2025.csv"

print(f"⏳ Iniciando geração de {len(MONTHS) * N_EMPLOYEES:,} registros para o período 2023-2025...")

# 1. Estrutura Geográfica e Custos (USD)
states_regions = {
    'Northeast': ['NY', 'MA', 'PA', 'NJ', 'CT'],
    'Midwest':   ['IL', 'OH', 'MI', 'IN', 'WI'],
    'South':     ['TX', 'FL', 'GA', 'NC', 'VA'],
    'West':      ['CA', 'WA', 'AZ', 'CO', 'OR']
}
all_states = [state for sublist in states_regions.values() for state in sublist]
region_map = {state: region for region, states in states_regions.items() for state in states}
depts = ['Engineering', 'Sales', 'Customer Success', 'Operations', 'HR', 'Finance']
roles = ['Associate', 'Specialist', 'Senior', 'Manager', 'Director']
cost_map = {'Associate': 5500, 'Specialist': 8000, 'Senior': 12000, 'Manager': 16500, 'Director': 24000}

# 2. Criar Base de Funcionários (Dados Mestres)
emp_ids = np.arange(100000, 100000 + N_EMPLOYEES)
emp_states = np.random.choice(all_states, N_EMPLOYEES)
emp_depts = np.random.choice(depts, N_EMPLOYEES)
emp_roles = np.random.choice(roles, N_EMPLOYEES, p=[0.4, 0.3, 0.15, 0.1, 0.05])

# Datas de contratação: Distribuídas entre 2020 e o meio de 2024
start_hiring = pd.Timestamp('2020-01-01').value
end_hiring = pd.Timestamp('2024-06-01').value
emp_hire_dates = pd.to_datetime(np.random.randint(start_hiring, end_hiring, N_EMPLOYEES, dtype=np.int64))

# Turnover: Simulando 8% de saídas distribuídas ao longo dos 3 anos
emp_status_final = np.random.choice(['Active', 'Terminated'], N_EMPLOYEES, p=[0.92, 0.08])
emp_term_dates = [MONTHS[np.random.randint(0, len(MONTHS))] if s == 'Terminated' else None for s in emp_status_final]

df_base = pd.DataFrame({
    'employee_id': emp_ids,
    'state': emp_states,
    'department': emp_depts,
    'role': emp_roles,
    'hire_date': emp_hire_dates,
    'termination_date': emp_term_dates
})
df_base['region'] = df_base['state'].map(region_map)

# 3. Geração dos Snapshots Mensais
data_frames = []

for month in MONTHS:
    # Filtrar apenas quem estava na empresa no mês de referência
    # Regra: Hire Date <= mês E (Term Date é nula OU Term Date >= mês)
    mask = (df_base['hire_date'] <= month) & ((df_base['termination_date'].isna()) | (df_base['termination_date'] >= month))
    temp_df = df_base[mask].copy()

    n_active = len(temp_df)
    temp_df['period'] = month
    temp_df['planned_hours'] = 160.0

    # Métricas variáveis (Utilização e Produtividade)
    util_factor = np.random.normal(0.94, 0.08, n_active).clip(0.6, 1.2)
    temp_df['worked_hours'] = (temp_df['planned_hours'] * util_factor).round(1)
    temp_df['overtime_hours'] = (temp_df['worked_hours'] - 160).clip(lower=0).round(1)

    # Financeiro e Metas
    temp_df['monthly_cost'] = (temp_df['role'].map(cost_map) * np.random.uniform(0.98, 1.02, n_active)).round(2)
    temp_df['goal_achievement'] = np.random.uniform(70, 105, n_active).round(1)

    # Status específico do mês
    is_term_month = (temp_df['termination_date'] == month)
    temp_df['status_in_month'] = np.where(is_term_month, 'Terminated', 'Active')

    data_frames.append(temp_df)

# Concatenar todos os meses
df_final = pd.concat(data_frames, ignore_index=True)

# 4. Seleção de Colunas e Exportação
cols_final = [
    'period', 'employee_id', 'region', 'state', 'department', 'role',
    'monthly_cost', 'planned_hours', 'worked_hours', 'overtime_hours',
    'goal_achievement', 'status_in_month'
]

# Exportar para CSV
df_final[cols_final].to_csv(FILE_NAME, index=False)

print(f"---")
print(f"✅ Arquivo gerado: {FILE_NAME}")
print(f"📊 Volume de dados: {len(df_final):,} linhas")
print(f"📅 Período: {df_final['period'].min().date()} a {df_final['period'].max().date()}")

⏳ Iniciando geração de 3,600,000 registros para o período 2023-2025...
---
✅ Arquivo gerado: globalforce_usa_3years_2023_2025.csv
📊 Volume de dados: 3,175,658 linhas
📅 Período: 2023-01-01 a 2025-12-01
